# Advanced Python Internals

*All Dunder Methods · Descriptor Protocol · __slots__ · Metaclasses · Typing & Protocols · CPython Memory Model · Real-World*


---
## Introduction


# Advanced Dunder

*Run each cell with **Shift+Enter***

00 — Python Mastery: Advanced Dunder (Magic) Methods
====================================================

Runnable companion to PDF Book I "Advanced Python — the data model".

Dunder ("double underscore") methods are how your objects plug into Python's
built-in syntax. Implement them and your class behaves like a native type:
it prints nicely, compares, hashes, iterates, indexes, works in `with`, and
can even be called like a function. Each demo below is self-checking.

Covered:
  * __repr__ / __str__      — developer vs user string
  * __eq__ / __hash__       — value equality + usability as a dict/set key
  * __lt__ (+ total_ordering) — ordering/sorting
  * __len__ / __getitem__ / __contains__ — behave like a container
  * __iter__ / __next__     — be iterable in a for-loop
  * __enter__ / __exit__    — be a context manager
  * __call__                — be callable like a function
  * __getattr__             — dynamic attribute fallback
  * __add__                 — operator overloading


---
## 🧠 Notebook Mental Model: Advanced Python Internals

> **Think of dunder methods as Python's "hook system" — implement a hook and your object joins any protocol for free.**  
> Python doesn't require inheritance from a base class; it just calls the right dunder method if it exists.

### The Python Data Model — One Diagram

```
YOUR CLASS
    │
    ├─ __repr__ / __str__       → how it prints
    ├─ __eq__ / __hash__        → equality + usability as dict/set key
    ├─ __lt__ + @total_ordering → sorting
    ├─ __len__ / __getitem__    → container protocol
    ├─ __iter__ / __next__      → iteration protocol
    ├─ __enter__ / __exit__     → context manager protocol
    ├─ __call__                 → callable protocol
    ├─ __add__ / __mul__ / …    → operator overloading
    └─ __getattr__ / __setattr__→ dynamic attribute access

Implement any hook → your class works with the corresponding built-in syntax.
```

### Why / What / How / When Summary

| Feature | WHY | WHAT | HOW | WHEN |
|---------|-----|------|-----|------|
| **Dunder methods** | Plug into Python's built-in syntax | Protocol methods starting/ending with `__` | Python calls them automatically | Whenever you need native-feeling objects |
| **`__slots__`** | Save memory; restrict attributes | Replaces `__dict__` with a fixed layout | `__slots__ = ("x", "y")` in class body | Millions of tiny objects |
| **Descriptor protocol** | Reusable attribute logic | `__get__`, `__set__`, `__delete__` on a class | Assigned as a class attribute | Type validators, lazy properties, ORM fields |
| **Metaclass** | Control class creation | `type` subclass used as `metaclass=` | `__new__`, `__init__`, `__prepare__` on metaclass | Plugin registries, framework hooks |
| **`typing.Protocol`** | Structural subtyping (duck typing with type hints) | Interface without inheritance | `class P(Protocol): def method(): ...` | Libraries that want type safety without coupling |

### The Descriptor Protocol — How `property` Works Internally

```
class property:          # roughly how @property is implemented
    def __get__(self, obj, cls):
        if obj is None: return self   # accessed on class → return descriptor
        return self.fget(obj)         # accessed on instance → call getter

    def __set__(self, obj, value):
        self.fset(obj, value)         # call setter

    def __delete__(self, obj):
        self.fdel(obj)                # call deleter

# So:
class C:
    @property
    def x(self): return self._x     # C.x is a DESCRIPTOR object in C.__dict__
                                    # c.x triggers C.x.__get__(c, C)
```

### Memory Layout: Normal class vs `__slots__`

```
Normal class instance:
  instance → PyObject header + __dict__ (hash table, ~232 bytes overhead)

__slots__ class instance:
  instance → PyObject header + fixed array of slot values (~56 bytes overhead)

Saving: ~50–70% per instance
Trade-off: no dynamic attributes; no multiple inheritance with conflicting slots
```


In [ ]:
from functools import total_ordering

---
## __repr__/__str__, __eq__/__hash__, ordering — a well-behaved value object.


In [ ]:
@total_ordering
class Money:
    """An immutable value object: two Moneys are equal by value and orderable."""

    __slots__ = ("cents",)  # (also demonstrates slots: no per-instance __dict__)

    def __init__(self, dollars: float):
        object.__setattr__(self, "cents", round(dollars * 100))

    # developer-facing, unambiguous; ideally eval-able
    def __repr__(self) -> str:
        return f"Money({self.cents / 100:.2f})"

    # user-facing, pretty
    def __str__(self) -> str:
        return f"${self.cents / 100:,.2f}"

    # value equality — two Money with same cents are equal
    def __eq__(self, other) -> bool:
        return isinstance(other, Money) and self.cents == other.cents

    # equal objects MUST hash equal -> safe as dict/set keys
    def __hash__(self) -> int:
        return hash(self.cents)

    # total_ordering derives <=, >, >= from __eq__ + __lt__
    def __lt__(self, other) -> bool:
        return self.cents < other.cents

    # operator overloading
    def __add__(self, other) -> "Money":
        return Money((self.cents + other.cents) / 100)


def dunder_value_object() -> None:
    a, b = Money(19.99), Money(19.99)
    assert a == b                      # __eq__
    assert hash(a) == hash(b)          # equal -> same hash
    assert {a} == {b}                  # usable as set members
    assert Money(5) < Money(10)        # __lt__
    assert Money(10) >= Money(10)      # derived by total_ordering
    assert sorted([Money(3), Money(1), Money(2)]) == [Money(1), Money(2), Money(3)]
    assert (Money(1.50) + Money(2.50)) == Money(4.00)   # __add__
    assert repr(Money(4)) == "Money(4.00)"
    assert str(Money(1234.5)) == "$1,234.50"
    print("   value object: __repr__/__eq__/__hash__/__lt__/__add__ all behave natively")

---
## Container protocol: __len__, __getitem__, __contains__, __iter__.


In [ ]:
class Playlist:
    def __init__(self, songs):
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, i):          # enables indexing AND slicing AND iteration
        return self._songs[i]

    def __contains__(self, song):      # enables `in`
        return song in self._songs


def dunder_container() -> None:
    p = Playlist(["a", "b", "c", "d"])
    assert len(p) == 4                 # __len__
    assert p[0] == "a"                 # __getitem__
    assert p[1:3] == ["b", "c"]        # slicing via __getitem__
    assert "c" in p                    # __contains__
    assert [s.upper() for s in p] == ["A", "B", "C", "D"]  # iterable via __getitem__
    print("   container: len(), indexing, slicing, `in`, and iteration all work")

---
## Iterator protocol: __iter__ returns an object with __next__.


In [ ]:
class Countdown:
    def __init__(self, start: int):
        self.start = start

    def __iter__(self):
        self._n = self.start
        return self

    def __next__(self):
        if self._n <= 0:
            raise StopIteration
        self._n -= 1
        return self._n + 1


def dunder_iterator() -> None:
    assert list(Countdown(3)) == [3, 2, 1]
    got = [n for n in Countdown(5)]
    assert got == [5, 4, 3, 2, 1]
    print("   iterator: __iter__/__next__ drive a real for-loop (StopIteration ends it)")

---
## Context manager: __enter__/__exit__ guarantee cleanup even on exceptions.


In [ ]:
class Transaction:
    def __init__(self, log):
        self.log = log

    def __enter__(self):
        self.log.append("BEGIN")
        return self

    def __exit__(self, exc_type, exc, tb):
        # returning False re-raises any exception; we roll back on error
        self.log.append("ROLLBACK" if exc_type else "COMMIT")
        return False


def dunder_context_manager() -> None:
    log = []
    with Transaction(log):
        log.append("work")
    assert log == ["BEGIN", "work", "COMMIT"]

    log2 = []
    try:
        with Transaction(log2):
            log2.append("work")
            raise ValueError("boom")
    except ValueError:
        pass
    assert log2 == ["BEGIN", "work", "ROLLBACK"]   # cleanup ran despite the error
    print("   context manager: __enter__/__exit__ commit on success, roll back on error")

---
## __call__ (callable instances) and __getattr__ (dynamic fallback).


In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):             # instance behaves like a function
        return x * self.factor


class Config:
    def __init__(self, data):
        self._data = data

    def __getattr__(self, name):       # only called when normal lookup FAILS
        try:
            return self._data[name]
        except KeyError:
            raise AttributeError(name) from None


def dunder_callable_and_getattr() -> None:
    triple = Multiplier(3)
    assert triple(10) == 30            # __call__
    assert callable(triple)
    cfg = Config({"host": "localhost", "port": 5432})
    assert cfg.host == "localhost"     # __getattr__ fallback into the dict
    assert cfg.port == 5432
    try:
        _ = cfg.missing
        raise AssertionError("should have raised")
    except AttributeError:
        pass
    print("   __call__: instances act like functions; __getattr__: dynamic attribute access")


def main() -> None:
    print("=" * 68)
    print("PYTHON MASTERY — advanced_dunder.py")
    print("=" * 68)
    print("1. Value object (repr/eq/hash/ordering/add):")
    dunder_value_object()
    print("2. Container protocol (len/getitem/contains/iter):")
    dunder_container()
    print("3. Iterator protocol (iter/next):")
    dunder_iterator()
    print("4. Context manager (enter/exit):")
    dunder_context_manager()
    print("5. Callable + dynamic attributes (call/getattr):")
    dunder_callable_and_getattr()
    print("-" * 68)
    print("All advanced_dunder demos passed ✔")

## ═══  EXHAUSTIVE DUNDER GOTCHA NOTEBOOK  ═══════════════════════════════════

In [ ]:
import sys, copy, math

def sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")


def notebook_repr_str() -> None:

## §D1 · __repr__ vs __str__ — Which Gets Called When?

In [ ]:
# __repr__: "developer" string; goal: eval()-able; called by repr(), interactive REPL.
    # __str__:  "user" string; called by str(), print().
    # Fallback chain:
    #   str(obj)   → obj.__str__() → obj.__repr__() (if no __str__)
    #   repr(obj)  → obj.__repr__() (never falls back to __str__)
    #   f"{obj}"   → obj.__format__("") → obj.__str__() → obj.__repr__()
    #   print(obj) → str(obj)
    #
    # GOTCHA 1: If you only define __repr__, str() and print() ALSO use it (fallback).
    # GOTCHA 2: If you only define __str__, repr() and the REPL still use the default.
    # GOTCHA 3: __repr__ should ideally be eval()-able: eval(repr(obj)) == obj.

    class OnlyRepr:
        def __repr__(self): return "OnlyRepr()"

    class OnlyStr:
        def __str__(self): return "human-readable"

    class Both:
        def __repr__(self): return "Both(repr)"
        def __str__(self):  return "Both(str)"

    obj_r = OnlyRepr()
    obj_s = OnlyStr()
    obj_b = Both()

    print(f"OnlyRepr — repr(): {repr(obj_r)!r}")          # 'OnlyRepr()'
    print(f"OnlyRepr — str():  {str(obj_r)!r}")           # 'OnlyRepr()' ← fallback to __repr__

    print(f"OnlyStr  — repr(): {repr(obj_s)!r}")           # '<__main__.OnlyStr ...>' ← default
    print(f"OnlyStr  — str():  {str(obj_s)!r}")            # 'human-readable'

    print(f"Both     — repr(): {repr(obj_b)!r}")           # 'Both(repr)'
    print(f"Both     — str():  {str(obj_b)!r}")            # 'Both(str)'
    print(f"Both     — f-str: {'both: ' + str(obj_b)!r}")  # uses __str__

    # GOTCHA 3: eval()-able repr
    class Point:
        def __init__(self, x, y): self.x, self.y = x, y
        def __repr__(self): return f"Point({self.x}, {self.y})"
        def __eq__(self, o): return isinstance(o, Point) and (self.x,self.y)==(o.x,o.y)

    p = Point(3, 4)
    print(f"eval(repr(p)) == p: {eval(repr(p)) == p}")   # True ← good repr


def notebook_eq_hash_contract() -> None:

## §D2 · __eq__ / __hash__ — The Dangerous Contract

In [ ]:
# CONTRACT: if a == b then hash(a) == hash(b)
    # INVERSE: hash(a) == hash(b) does NOT imply a == b (collision is allowed)
    #
    # GOTCHA 1: Override __eq__ without __hash__ → Python sets __hash__=None
    #           → instances become UNHASHABLE (can't be dict key or set member)
    # GOTCHA 2: Override both inconsistently or MUTATE a key after insertion
    #           → dict/set lookup silently MISSES the entry (data corruption)
    # GOTCHA 3: equal objects from different types CAN share a hash
    #           (1 == 1.0 == True → hash(1)==hash(1.0)==hash(True)==1)

    # GOTCHA 1
    class BadKey:
        def __eq__(self, other): return True   # __hash__ becomes None!

    bk = BadKey()
    try:
        {bk}                        # TypeError: unhashable type
    except TypeError as e:
        print(f"GOTCHA 1 — no __hash__: {e}")

    # GOTCHA 2 — mutable key corrupts dict
    class MutableKey:
        def __init__(self, v): self.v = v
        def __eq__(self, o): return isinstance(o, MutableKey) and self.v == o.v
        def __hash__(self):  return hash(self.v)

    mk = MutableKey(10)
    d = {mk: "found"}
    print(f"Before mutation: d[mk] = {d.get(mk, 'MISS')!r}")   # 'found'
    mk.v = 99                       # mutate the key AFTER insertion!
    print(f"After mutation:  d[mk] = {d.get(mk, 'MISS')!r}")   # 'MISS' — hash changed!
    # The entry still exists in memory but is unreachable:
    print(f"d has {len(d)} entry but key is unfindable — memory leak!")

    # GOTCHA 3 — cross-type equality
    print(f"1 == 1.0 == True: {1==1.0==True}")
    print(f"hash(1)==hash(1.0)==hash(True): {hash(1)==hash(1.0)==hash(True)}")
    mixed = {1: "int", True: "overwritten"}   # True IS 1, so it overwrites!
    print(f"{{1:'int', True:'overwritten'}} = {mixed}")   # {1: 'overwritten'}


def notebook_ordering() -> None:

## §D3 · __lt__ + @total_ordering — Gotchas

In [ ]:
# @total_ordering derives __le__, __gt__, __ge__ from __lt__ + __eq__.
    # GOTCHA 1: Without @total_ordering you only get __lt__; sorted() works
    #           but <=, >=, > raise TypeError.
    # GOTCHA 2: @total_ordering is ~2–3× slower per comparison vs defining all 6.
    #           Use it unless the class is a hot path.
    # GOTCHA 3: NotImplemented (not raise TypeError) signals "I can't handle this type".

    class NoOrdering:
        def __init__(self, v): self.v = v
        def __eq__(self, o): return self.v == o.v
        def __lt__(self, o): return self.v < o.v
        def __hash__(self):  return hash(self.v)

    a, b = NoOrdering(1), NoOrdering(2)
    assert a < b and sorted([b, a]) == [a, b]   # __lt__ works
    try:
        a <= b                          # TypeError — __le__ not defined!
    except TypeError as e:
        print(f"GOTCHA 1 — no @total_ordering: {e}")

    # Fix: add @total_ordering
    from functools import total_ordering

    @total_ordering
    class Ordered:
        def __init__(self, v): self.v = v
        def __eq__(self, o): return isinstance(o,Ordered) and self.v==o.v
        def __lt__(self, o): return self.v < o.v
        def __hash__(self):  return hash(self.v)

    x, y = Ordered(1), Ordered(2)
    assert x < y and x <= y and y > x and y >= x
    print("@total_ordering: <, <=, >, >= all work ✓")

    # GOTCHA 3 — return NotImplemented for unknown types
    @total_ordering
    class Safe:
        def __init__(self, v): self.v = v
        def __eq__(self, o):
            if not isinstance(o, Safe): return NotImplemented
            return self.v == o.v
        def __lt__(self, o):
            if not isinstance(o, Safe): return NotImplemented
            return self.v < o.v
        def __hash__(self): return hash(self.v)

    s = Safe(5)
    result = s.__eq__("not-a-Safe")
    print(f"comparison with wrong type returns: {result!r}")   # NotImplemented
    # Python then tries the reflected operation on "not-a-Safe".__eq__(s)
    # If both return NotImplemented, Python falls back to identity comparison.


def notebook_container_protocol() -> None:

## §D4 · Container Protocol — All Dunders

In [ ]:
# __len__      → len(obj)
    # __getitem__  → obj[key], slicing, and implicit iteration (no __iter__ needed)
    # __setitem__  → obj[key] = val
    # __delitem__  → del obj[key]
    # __contains__ → `in` operator  (falls back to O(n) scan via __iter__/__getitem__)
    # __missing__  → called by dict subclass when key not found
    #
    # GOTCHA 1: Defining __getitem__ alone (no __iter__) makes a class iterable!
    #           Python generates iteration by calling __getitem__(0), (1), … until IndexError.
    # GOTCHA 2: __contains__ defaults to O(n) scan through __iter__; override for O(1).
    # GOTCHA 3: __missing__ is only called by dict SUBCLASSES, not plain dict.

    class SliceableLog:
        def __init__(self, data): self._data = list(data)
        def __len__(self):           return len(self._data)
        def __getitem__(self, i):    return self._data[i]   # supports slicing automatically
        def __setitem__(self, i, v): self._data[i] = v
        def __delitem__(self, i):    del self._data[i]
        def __contains__(self, x):  return x in self._data  # O(n) scan — could override

    log = SliceableLog([10, 20, 30, 40, 50])
    print(f"len={len(log)}  log[1]={log[1]}  log[1:3]={log[1:3]}")
    log[0] = 99
    del log[0]
    print(f"After set/del: {list(log)}")
    print(f"30 in log: {30 in log}")

    # GOTCHA 1: __getitem__ alone enables for-loop (no __iter__ needed)
    class Counted:
        def __init__(self, n): self.n = n
        def __getitem__(self, i):
            if i >= self.n: raise IndexError
            return i * 2
        # no __iter__ defined!

    c = Counted(4)
    print(f"iteration via __getitem__ only: {list(c)}")   # [0,2,4,6]
    print(f"4 in Counted(5): {4 in Counted(5)}")          # True (O(n) scan)

    # GOTCHA 3: __missing__ only for dict subclasses
    class AutoDict(dict):
        def __missing__(self, key):
            self[key] = key.upper()   # auto-create
            return self[key]

    ad = AutoDict()
    print(f"ad['hello'] = {ad['hello']!r}")   # 'HELLO' — __missing__ called
    print(f"ad now has: {dict(ad)}")           # {'hello': 'HELLO'}
    # On a plain dict, __missing__ is NEVER called; KeyError is raised directly.


def notebook_context_manager_protocol() -> None:

## §D5 · Context Manager __enter__/__exit__ Gotchas

In [ ]:
# GOTCHA 1: __exit__ returning a TRUTHY value SUPPRESSES any exception.
    #           This is almost always wrong and a very hard-to-find bug.
    # GOTCHA 2: The exception is passed as (type, value, traceback).
    #           Returning False/None lets it propagate.
    # GOTCHA 3: __enter__ return value is bound to the `as` target.
    #           Returning self is idiomatic; returning a different object is valid.
    # GOTCHA 4: If __enter__ raises, __exit__ is NOT called (no cleanup needed —
    #           nothing was acquired yet).

    # GOTCHA 1: accidental suppression
    class SuppressingCM:
        def __enter__(self): return self
        def __exit__(self, *_): return True   # ALL exceptions silently suppressed!

    try:
        with SuppressingCM():
            raise ValueError("this should propagate")
        print("GOTCHA 1: exception was SUPPRESSED by __exit__ returning True")
    except ValueError:
        print("This line never runs when __exit__ returns True")

    # Correct: return False (or None) to propagate
    class SafeCM:
        def __enter__(self): self.log = ["BEGIN"]; return self
        def __exit__(self, exc_type, exc, tb):
            self.log.append("FAIL" if exc_type else "OK")
            return False   # ← never suppress

    log_obj = None
    try:
        with SafeCM() as cm:
            log_obj = cm
            raise RuntimeError("test")
    except RuntimeError:
        pass
    print(f"SafeCM log: {log_obj.log}")   # ['BEGIN', 'FAIL'] — cleanup ran despite exception

    # GOTCHA 4: if __enter__ raises, __exit__ is NOT called
    class FailEnter:
        entered = False
        exited  = False
        def __enter__(self):
            raise ConnectionError("cannot connect")
        def __exit__(self, *_):
            FailEnter.exited = True   # this NEVER runs if __enter__ raised

    try:
        with FailEnter(): pass
    except ConnectionError:
        pass
    print(f"__exit__ called when __enter__ raises: {FailEnter.exited}")   # False


def notebook_arithmetic_dunders() -> None:

## §D6 · Arithmetic Dunders + NotImplemented Pattern

In [ ]:
# For binary operations (a + b):
    #   Python tries: type(a).__add__(a, b)
    #   If it returns NotImplemented, then tries: type(b).__radd__(b, a)
    #   If both return NotImplemented: TypeError
    #
    # GOTCHA 1: Raising TypeError instead of returning NotImplemented breaks
    #           the reflected-operation protocol.
    # GOTCHA 2: __iadd__ (+=) should modify in-place and return self for mutables.
    #           If __iadd__ is missing, Python falls back to __add__ which creates a new object.
    # GOTCHA 3: __mul__ and __rmul__ let you do  scalar * obj  as well as  obj * scalar.

    class Vector:
        def __init__(self, *coords): self.coords = coords

        def __repr__(self): return f"Vector{self.coords}"

        def __add__(self, other):
            if not isinstance(other, Vector):
                return NotImplemented       # ← not raise TypeError!
            return Vector(*(a+b for a,b in zip(self.coords, other.coords)))

        def __mul__(self, scalar):
            if not isinstance(scalar, (int, float)):
                return NotImplemented
            return Vector(*(c*scalar for c in self.coords))

        def __rmul__(self, scalar):         # scalar * v → type(scalar).__mul__ fails → try v.__rmul__
            return self.__mul__(scalar)

        def __iadd__(self, other):
            if not isinstance(other, Vector):
                return NotImplemented
            self.coords = tuple(a+b for a,b in zip(self.coords, other.coords))
            return self                     # MUST return self

        def __eq__(self, o): return isinstance(o,Vector) and self.coords==o.coords
        def __hash__(self):  return hash(self.coords)

    v1 = Vector(1, 2, 3)
    v2 = Vector(4, 5, 6)
    print(f"v1 + v2  = {v1 + v2}")        # Vector(5,7,9)
    print(f"v1 * 3   = {v1 * 3}")          # Vector(3,6,9)
    print(f"3 * v1   = {3 * v1}")          # Vector(3,6,9) — via __rmul__

    v1 += v2                               # calls __iadd__, modifies v1 in-place
    print(f"v1 after +=: {v1}")            # Vector(5,7,9)

    # GOTCHA: if __iadd__ missing, += creates NEW object via __add__
    class NoIadd:
        def __init__(self, v): self.v = v
        def __add__(self, o): return NoIadd(self.v + o.v)
        def __eq__(self, o): return self.v==o.v
        def __hash__(self): return hash(self.v)

    a = NoIadd(1)
    old_id = id(a)
    a += NoIadd(2)   # falls back to __add__; rebinds a to a NEW object
    print(f"+= without __iadd__ creates new object: {id(a) != old_id}")   # True


def notebook_getattr_vs_getattribute() -> None:

## §D7 · __getattr__ vs __getattribute__ — Critical Difference

In [ ]:
# __getattr__(self, name):
    #   Called ONLY when the attribute is NOT found by normal means.
    #   Safe fallback; does NOT intercept existing attributes.
    #
    # __getattribute__(self, name):
    #   Called for EVERY attribute access, including existing ones.
    #   GOTCHA: calling self.anything inside __getattribute__ recurses infinitely!
    #   Fix: use object.__getattribute__(self, name) to bypass.
    #
    # GOTCHA: Defining __getattr__ does NOT intercept attribute-not-found for
    #         special dunders (Python looks those up on the TYPE, not the instance).

    class SafeFallback:
        def __init__(self, data): self._data = data
        def __getattr__(self, name):                  # only for MISSING attrs
            try:    return self._data[name]
            except KeyError: raise AttributeError(name) from None

    cfg = SafeFallback({"host": "localhost", "port": 5432})
    print(f"cfg.host = {cfg.host}")    # 'localhost' via __getattr__
    print(f"cfg._data is direct: {cfg._data}")  # direct — __getattr__ NOT called
    try:
        _ = cfg.missing
    except AttributeError as e:
        print(f"Missing key → AttributeError: {e}")

    class InterceptAll:
        _log = []
        def __getattribute__(self, name):
            InterceptAll._log.append(name)
            return object.__getattribute__(self, name)  # ← MUST use object.__getattribute__
            # If you wrote `return self.__dict__[name]` you'd recurse infinitely!

        def __init__(self): self.x = 42

    ia = InterceptAll()
    _ = ia.x
    print(f"__getattribute__ intercepts everything; logged: {InterceptAll._log[-3:]}")


def notebook_slots_and_weakref_interaction() -> None:

## §D8 · __slots__ + weakref + inheritance Gotchas

In [ ]:
# GOTCHA 1: __slots__ removes __dict__ AND __weakref__ by default.
    #           A slotted class cannot be weakly referenced unless
    #           '__weakref__' is listed in __slots__.
    # GOTCHA 2: A subclass of a slotted class that doesn't define __slots__
    #           gets __dict__ back — negating the memory savings.
    # GOTCHA 3: Multiple inheritance with __slots__ only works cleanly when
    #           ALL bases are slotted (otherwise __dict__ re-appears anyway).

    import weakref

    class NoWeakref:
        __slots__ = ("x",)
        def __init__(self, x): self.x = x

    class WithWeakref:
        __slots__ = ("x", "__weakref__")
        def __init__(self, x): self.x = x

    try:
        weakref.ref(NoWeakref(1))
    except TypeError as e:
        print(f"GOTCHA 1 — no weakref slot: {e}")

    wr = weakref.ref(WithWeakref(1))
    print(f"weakref works with '__weakref__' in __slots__: {wr()!r}")

    # GOTCHA 2: subclass without __slots__ regains __dict__
    class Slotted:
        __slots__ = ("x",)
        def __init__(self): self.x = 0

    class SubNoSlots(Slotted):
        pass   # no __slots__ defined!

    s  = Slotted()
    sub = SubNoSlots()
    print(f"Slotted  has __dict__: {hasattr(s,   '__dict__')}")    # False ✓
    print(f"SubNoSlots has __dict__: {hasattr(sub, '__dict__')}")  # True — memory savings lost!
    sub.new_attr = 42   # subclass can now have dynamic attrs


def notebook_descriptor_interaction_with_dunders() -> None:

## §D9 · How Dunders Are Looked Up — Type vs Instance

In [ ]:
# Special (dunder) methods are looked up on the TYPE, NOT the instance.
    # GOTCHA: Assigning __repr__ to an INSTANCE has no effect on repr().
    #         Python only looks up dunders on type(obj).__repr__.

    class Widget:
        def __repr__(self): return "Widget(default)"

    w = Widget()
    w.__repr__ = lambda: "Widget(instance-level)"   # assign to instance
    print(f"repr(w)  = {repr(w)!r}")                # 'Widget(default)' ← type wins!
    print(f"w.__repr__() = {w.__repr__()!r}")       # 'Widget(instance-level)' ← explicit call

    # This is why operator dispatch is safe:
    # Even if you monkey-patch an instance, operators use the type's dunder.

    # Practical implication: you can proxy dunders through __class__ reassignment:
    class Proxy:
        def __init__(self, wrapped):
            self.__dict__ = wrapped.__dict__
            self._wrapped = wrapped

    p = Proxy(Widget())
    print(f"Proxy repr (uses Proxy's __repr__): {repr(p)!r}")   # default object repr


def run_dunder_notebook() -> None:
    notebook_repr_str()
    notebook_eq_hash_contract()
    notebook_ordering()
    notebook_container_protocol()
    notebook_context_manager_protocol()
    notebook_arithmetic_dunders()
    notebook_getattr_vs_getattribute()
    notebook_slots_and_weakref_interaction()
    notebook_descriptor_interaction_with_dunders()
    print("\n" + "═"*64)
    print("  DUNDER NOTEBOOK COMPLETE — all gotchas demonstrated")
    print("═"*64)


if __name__ == "__main__":
    # Keep Unicode output safe even when stdout is redirected/piped (Windows cp1252 fallback).
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_dunder_notebook()

---
## Descriptors, __slots__, Metaclasses


# Descriptors Slots Meta

*Run each cell with **Shift+Enter***

00 — Python Mastery: Descriptors, __slots__, Decorators-with-args & Metaclasses
==============================================================================

Runnable companion to PDF Book I "Advanced Python — how classes really work".

These are the "how does Python itself work?" features senior interviews probe:

  * DESCRIPTORS  — objects with __get__/__set__ that control attribute access.
                   This is the machinery behind @property, methods, classmethod,
                   staticmethod, and ORM fields. We build a validating one.
  * __slots__    — trade the per-instance __dict__ for a fixed set of fields:
                   less memory, faster access, no accidental new attributes.
  * DECORATOR    — a decorator that TAKES ARGUMENTS (a 3-level nesting) plus a
    FACTORY        class-based decorator; the general pattern behind retry(n),
                   lru_cache(maxsize=...), app.get("/path"), etc.
  * METACLASS    — a class whose instances are classes; customizes class
                   CREATION. We build a registry metaclass (the plugin pattern).

In [ ]:
import functools

---
## DESCRIPTOR: reusable, validating managed attribute. __set_name__ learns the



### 🧠 Mental Model: Descriptor Protocol

**WHY** — Descriptors are the mechanism that powers `@property`, `classmethod`, `staticmethod`, and ORM field validators. Understanding descriptors means understanding how Python attribute access *really* works.

**WHAT** — A descriptor is a class that defines `__get__`, `__set__`, or `__delete__`. When assigned as a *class* attribute, these methods intercept all attribute access on instances.

**HOW — attribute lookup order (`obj.attr`):**
```
1. DATA descriptor in type(obj).__mro__     ← __get__ + (__set__ OR __delete__)
2. obj.__dict__["attr"]                      ← instance dictionary
3. NON-DATA descriptor in type(obj).__mro__ ← __get__ only
4. AttributeError

DATA descriptor wins over instance dict → can't be shadowed
NON-DATA descriptor loses to instance dict → CAN be shadowed
```

**Non-data vs Data descriptor:**
```python
class NonData:          # only __get__ → instance dict CAN shadow it
    def __get__(self, obj, cls): return "value"

class Data:             # __get__ + __set__ → instance dict CANNOT shadow it
    def __get__(self, obj, cls): ...
    def __set__(self, obj, val): ...

# @property is a DATA descriptor (has __get__ + __set__ + __delete__)
# Instance methods are NON-DATA descriptors (have __get__ only)
```

**`__set_name__` — how the descriptor knows its own name:**
```python
class Validator:
    def __set_name__(self, owner, name):
        self._attr = "_" + name     # called when the class body is finished

class MyModel:
    age = Validator()   # Python calls Validator.__set_name__(MyModel, "age")
    # Now self._attr = "_age" — correct for this specific field
```

**WHEN to use descriptors:**
- Reusable attribute validators (type checking, range checking)
- ORM fields (`models.IntegerField()`)
- Lazy computed attributes (compute once and cache)
- Access control (audit logging every read/write)


In [ ]:
class Positive:
    """A data descriptor that rejects non-positive numbers."""

    def __set_name__(self, owner, name):
        self._name = "_" + name         # where we stash the real value

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self                 # accessed on the class, not an instance
        return getattr(obj, self._name)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self._name[1:]} must be positive, got {value}")
        setattr(obj, self._name, value)


class Product:
    price = Positive()                  # the descriptor manages `price`
    quantity = Positive()

    def __init__(self, price, quantity):
        self.price = price              # goes through Positive.__set__ (validates)
        self.quantity = quantity

    @property                           # @property is itself a descriptor
    def total(self):
        return self.price * self.quantity


def descriptor_demo() -> None:
    p = Product(price=10, quantity=3)
    assert p.total == 30                # @property computed attribute
    p.price = 20                        # validated on assignment
    assert p.total == 60
    for bad in (0, -5):
        try:
            Product(price=bad, quantity=1)
            raise AssertionError("should have rejected")
        except ValueError:
            pass
    print("   descriptor: one Positive() class validates every field it manages (DRY)")

---
## __slots__: fixed fields, no __dict__. Saves memory at scale and blocks typos.



### 🧠 Mental Model: `__slots__`

**WHY** — Every normal Python object carries a `__dict__` (a hash table) just to store its attributes. For millions of small objects, this is significant overhead (~200–300 bytes per instance). `__slots__` replaces the dict with a compact fixed-size C array.

**WHAT** — Declaring `__slots__` in a class tells Python: "this class will only ever have these specific attributes — don't allocate a `__dict__`."

**HOW:**
```python
class Point:
    __slots__ = ("x", "y")   # fixed attribute names

    def __init__(self, x, y):
        self.x = x    # stored in a slot (C array), NOT in __dict__
        self.y = y

p = Point(1, 2)
p.z = 3           # AttributeError — no __dict__, z not in slots
```

**Memory comparison (CPython 3.11):**
```
Normal instance:  sys.getsizeof(Normal()) ≈ 48 bytes + __dict__ ≈ 232 bytes = ~280 bytes
Slotted instance: sys.getsizeof(Slotted()) ≈ 56 bytes (no __dict__)

For 1 million instances: ~280 MB vs ~56 MB — 5× reduction
```

**WHEN to use `__slots__`:**
- Creating millions of small instances (particles, events, graph nodes)
- When you want to prevent accidental attribute creation (e.g., `obj.nmae = x` instead of `obj.name`)
- High-performance inner-loop code where attribute access speed matters

**Gotchas:**
```
1. Subclass without __slots__ → __dict__ comes back → savings lost
2. __weakref__ is removed by default; add it to __slots__ if needed
3. Multiple inheritance with __slots__: all bases must define __slots__
4. pickle needs __getstate__/__setstate__ for slotted classes
```


In [ ]:
class Slotted:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x, self.y = x, y


class Dicted:
    def __init__(self, x, y):
        self.x, self.y = x, y


def slots_demo() -> None:
    s = Slotted(1, 2)
    assert (s.x, s.y) == (1, 2)
    assert not hasattr(s, "__dict__")   # slots removed the per-instance dict
    try:
        s.z = 3                          # assigning an unknown attr is blocked
        raise AssertionError("slots should forbid new attributes")
    except AttributeError:
        pass
    d = Dicted(1, 2)
    d.z = 3                              # a normal class silently accepts typos
    assert d.__dict__ == {"x": 1, "y": 2, "z": 3}
    print("   __slots__: no per-instance __dict__ -> less memory, and typos raise instead of hiding")

---
## DECORATOR WITH ARGUMENTS: three nested functions -> deco(args)(fn)(*call).


In [ ]:
def retry(times: int):
    """Retry the wrapped function up to `times` attempts on exception."""

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last = None
            for _ in range(times):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:   # noqa: BLE001 - demo
                    last = e
            raise last
        return wrapper

    return decorator


def retry_demo() -> None:
    calls = {"n": 0}

    @retry(times=3)
    def flaky():
        calls["n"] += 1
        if calls["n"] < 3:
            raise ValueError("try again")
        return "ok"

    assert flaky() == "ok"
    assert calls["n"] == 3              # failed twice, succeeded on the third
    assert flaky.__name__ == "flaky"   # functools.wraps preserved identity
    print("   decorator-with-args: retry(times=3) is deco(args)(fn)(*call) — three layers")

---
## METACLASS: customize class CREATION. Here: auto-register every subclass in a



### 🧠 Mental Model: Metaclasses

**WHY** — Sometimes you need to control *class creation* itself, not just instance creation. Metaclasses let you add behaviour when a new class is defined (at class body parsing time), not when instances are created.

**WHAT** — A metaclass is the "class of a class". Just as `Dog()` creates an instance using `Dog.__init__`, `class Dog(Animal):` creates a class using `type.__call__` (the default metaclass). A custom metaclass replaces `type`.

**HOW — class creation steps:**
```
class MyClass(Base, metaclass=Meta):
    x = 1

# Python does:
namespace = Meta.__prepare__("MyClass", (Base,))   # 1. create namespace dict
# execute class body in namespace
cls = Meta.__new__(Meta, "MyClass", (Base,), namespace)  # 2. create class object
Meta.__init__(cls, "MyClass", (Base,), namespace)        # 3. initialise class
```

**WHEN metaclass vs alternatives:**
```
GOAL                              | BEST TOOL
----------------------------------|--------------------------------------------
Auto-register subclasses          | __init_subclass__ (simpler) or metaclass
Enforce interface at class def    | ABCs with @abstractmethod
Add methods to every subclass     | Metaclass __new__
Modify class namespace before def | Metaclass __prepare__ (advanced)
```

**`__init_subclass__` — 90% of metaclass use cases, simpler:**
```python
class Plugin:
    _registry = {}
    def __init_subclass__(cls, key="", **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin._registry[key or cls.__name__] = cls

class CsvExporter(Plugin, key="csv"): ...  # auto-registered
class JsonExporter(Plugin, key="json"): ... # auto-registered
```

**WHEN to use a real metaclass:**
- Framework-level magic (Django models, SQLAlchemy, pytest)
- When `__init_subclass__` is insufficient (need `__prepare__` for ordered attributes)
- When you need to intercept the class namespace before any code runs


In [ ]:
class PluginRegistry(type):
    registry = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:                        # skip the base class itself
            PluginRegistry.registry[name.lower()] = cls
        return cls


class Plugin(metaclass=PluginRegistry):
    pass


class JsonExporter(Plugin):
    def run(self):
        return "json"


class CsvExporter(Plugin):
    def run(self):
        return "csv"


def metaclass_demo() -> None:
    # Both subclasses registered themselves automatically at definition time.
    assert set(PluginRegistry.registry) == {"jsonexporter", "csvexporter"}
    exporter = PluginRegistry.registry["jsonexporter"]()
    assert exporter.run() == "json"
    print("   metaclass: subclasses auto-register at creation — the plugin pattern, zero boilerplate")


def main() -> None:
    print("=" * 70)
    print("PYTHON MASTERY — descriptors_slots_meta.py")
    print("=" * 70)
    print("1. Descriptors (the machinery behind @property/ORM fields):")
    descriptor_demo()
    print("2. __slots__ (memory + typo protection):")
    slots_demo()
    print("3. Decorator with arguments (3-level nesting):")
    retry_demo()
    print("4. Metaclass (customize class creation -> auto-registry):")
    metaclass_demo()
    print("-" * 70)
    print("All descriptors_slots_meta demos passed ✔")

## ═══  EXHAUSTIVE GOTCHA NOTEBOOK  ══════════════════════════════════════════

In [ ]:
import sys, weakref

def sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")


def notebook_descriptor_gotchas() -> None:

## §1 · Descriptor Protocol Gotchas

In [ ]:
# LOOKUP PRIORITY (obj.attr):
    #   1. DATA descriptor in type(obj).__mro__  (has __get__ + __set__ or __delete__)
    #   2. obj.__dict__["attr"]
    #   3. NON-DATA descriptor in type(obj).__mro__  (only __get__)
    #   4. AttributeError
    #
    # GOTCHA 1: A non-data descriptor CAN be shadowed by an instance attribute
    #           (because instance __dict__ beats non-data descriptor).
    # GOTCHA 2: A data descriptor CANNOT be shadowed — it always wins.
    # GOTCHA 3: Forgetting __set_name__ means self._name is None;
    #           the descriptor can't know which field it manages.
    # GOTCHA 4: Accessing a descriptor ON THE CLASS (not an instance) calls
    #           __get__(None, owner_class).  Return `self` there, not a value.

    class NonData:
        """Only __get__ — a non-data descriptor."""
        def __get__(self, obj, objtype=None):
            return "non-data value"

    class DataD:
        """__get__ + __set__ — a data descriptor."""
        def __get__(self, obj, objtype=None):
            if obj is None: return self          # class access → return self
            return obj.__dict__.get("_x", "data value")
        def __set__(self, obj, value):
            obj.__dict__["_x"] = value

    class MyClass:
        nd = NonData()
        dd = DataD()

    obj = MyClass()

    # GOTCHA 1: non-data descriptor can be shadowed by instance dict
    print(f"Before shadow  nd={obj.nd!r}")    # 'non-data value'
    obj.__dict__["nd"] = "SHADOWED"           # instance dict wins over non-data!
    print(f"After shadow   nd={obj.nd!r}")    # 'SHADOWED'

    # GOTCHA 2: data descriptor CANNOT be shadowed
    obj.__dict__["dd"] = "ATTEMPT"            # store in instance dict
    print(f"dd after shadow attempt: {obj.dd!r}")  # still 'data value' — data desc wins!

    # GOTCHA 3: __set_name__ not called → descriptor doesn't know its name
    class NoSetName:
        def __get__(self, obj, t=None):
            if obj is None: return self
            return getattr(obj, "_value", None)   # must hard-code name without __set_name__
        def __set__(self, obj, v):
            obj._value = v   # hard-coded "_value" — bug if reused under different names!

    class Owner:
        field1 = NoSetName()   # descriptors would conflict if both used "_value"
        field2 = NoSetName()

    o = Owner()
    o.field1 = 10
    o.field2 = 20   # OVERWRITES _value!  both fields share the same backing attr
    print(f"field1={o.field1}  field2={o.field2}  (both 20 — GOTCHA!)")

    # Fix: use __set_name__
    class WithSetName:
        def __set_name__(self, owner, name): self._attr = "_" + name
        def __get__(self, obj, t=None):
            return self if obj is None else getattr(obj, self._attr, None)
        def __set__(self, obj, v): setattr(obj, self._attr, v)

    class GoodOwner:
        field1 = WithSetName()
        field2 = WithSetName()

    g = GoodOwner()
    g.field1 = 10; g.field2 = 20
    print(f"GoodOwner field1={g.field1}  field2={g.field2}  ✓")

    # GOTCHA 4: class-level access
    print(f"MyClass.dd is the descriptor itself: {type(MyClass.dd).__name__}")
    print(f"MyClass.nd returns: {MyClass.nd!r}")   # 'non-data value' (no instance)


def notebook_slots_gotchas() -> None:

## §2 · __slots__ Gotchas

In [ ]:
# GOTCHA 1: Subclass without __slots__ re-adds __dict__, losing savings.
    # GOTCHA 2: __slots__ removes __weakref__ by default.
    # GOTCHA 3: Multiple inheritance: all bases must define __slots__ for it to work.
    # GOTCHA 4: pickle/copy may fail for slotted classes without __getstate__/__setstate__.
    # GOTCHA 5: __slots__ and @property can conflict if slot name == property name.

    class SlottedBase:
        __slots__ = ("x", "y")
        def __init__(self, x, y): self.x, self.y = x, y

    class SlottedChild(SlottedBase):
        __slots__ = ("z",)   # adds z; inherits x, y slots from parent
        def __init__(self, x, y, z): super().__init__(x, y); self.z = z

    class LazyChild(SlottedBase):
        pass   # NO __slots__ — __dict__ comes back!

    sc = SlottedChild(1, 2, 3)
    lc = LazyChild(4, 5)
    print(f"SlottedChild has __dict__: {hasattr(sc, '__dict__')}")  # False ✓
    print(f"LazyChild    has __dict__: {hasattr(lc, '__dict__')}")  # True (savings lost)

    # GOTCHA 2: weakref
    try:
        weakref.ref(SlottedBase(1, 2))
    except TypeError as e:
        print(f"GOTCHA 2 — no __weakref__: {e}")

    class SlottedWithWeakref:
        __slots__ = ("x", "__weakref__")
        def __init__(self, x): self.x = x

    wr = weakref.ref(SlottedWithWeakref(42))
    print(f"weakref with slot: {wr()!r}")

    # GOTCHA 4: pickle + __slots__
    # NOTE: Local classes (defined inside functions) cannot be pickled at all —
    # that is a separate pickle limitation.  In production, top-level slotted
    # classes need __getstate__/__setstate__ so pickle can find their values.
    import pickle

    class Unpicklable:
        __slots__ = ("val",)
        def __init__(self, v): self.val = v

    up = Unpicklable(99)
    try:
        data = pickle.dumps(up)
        restored = pickle.loads(data)
        print(f"Simple slotted class pickles OK: {restored.val}")
    except Exception as e:
        print(f"GOTCHA 4 — slotted pickle failed: {e}")
        print("  Fix: add __getstate__ / __setstate__ or use dataclass(slots=True)")

    print("  Pattern: __getstate__ returns a dict; __setstate__ restores from it")


def notebook_metaclass_gotchas() -> None:

## §3 · Metaclass Gotchas

In [ ]:
# GOTCHA 1: Only ONE metaclass can be active per class.
    #   If two base classes have DIFFERENT metaclasses, Python raises:
    #   TypeError: metaclass conflict: the metaclass of a derived class must be
    #   a (non-strict) subclass of the metaclasses of all its bases.
    # GOTCHA 2: Metaclass __new__ runs at CLASS DEFINITION time, not at
    #   instantiation time — any I/O or side effects run at import.
    # GOTCHA 3: super() in a metaclass needs explicit arguments
    #   because the zero-arg form may fail in __new__.
    # GOTCHA 4: __init_subclass__ is usually a simpler alternative to metaclass.

    # GOTCHA 1: metaclass conflict
    class MetaA(type): pass
    class MetaB(type): pass
    class A(metaclass=MetaA): pass
    class B(metaclass=MetaB): pass

    try:
        class C(A, B): pass   # MetaA and MetaB can't coexist
    except TypeError as e:
        print(f"GOTCHA 1 — metaclass conflict: {e}")

    # Fix: create a combined metaclass
    class MetaAB(MetaA, MetaB): pass
    class C2(A, B, metaclass=MetaAB): pass   # ✓
    print(f"Combined metaclass works: type(C2) = {type(C2).__name__}")

    # GOTCHA 2: side effects at class definition time
    log = []
    class SideEffectMeta(type):
        def __new__(mcs, name, bases, ns):
            log.append(f"defining {name}")   # runs at CLASS DEFINITION, not instance creation
            return super().__new__(mcs, name, bases, ns)

    class Monitored(metaclass=SideEffectMeta):   # log updated HERE, at import time
        pass

    print(f"Metaclass ran at class def time: {log}")   # ['defining Monitored']
    _ = Monitored()   # instantiation does NOT trigger metaclass __new__ again
    print(f"After instantiation: {log}")               # still ['defining Monitored']

    # GOTCHA 4: __init_subclass__ is usually simpler
    registry = {}
    class Plugin:
        def __init_subclass__(cls, key="", **kw):
            super().__init_subclass__(**kw)
            registry[key or cls.__name__.lower()] = cls

    class CsvPlugin(Plugin, key="csv"): pass
    class JsonPlugin(Plugin, key="json"): pass
    print(f"__init_subclass__ registry: {list(registry)}")   # ['csv','json']


def notebook_decorator_factory_gotchas() -> None:

## §4 · Decorator Factory (3-Level Nesting) Gotchas

In [ ]:
# GOTCHA 1: forgetting @functools.wraps erases __name__, __doc__, __module__.
    # GOTCHA 2: class-based decorator that stores state must be careful with
    #           method binding — the descriptor protocol applies.
    # GOTCHA 3: stacking decorators — they apply bottom-up but execute top-down.
    # GOTCHA 4: A parameterised decorator with no required args must be called:
    #   @retry()   NOT  @retry  (unless you write detection logic).

    import functools

    # GOTCHA 1: missing @wraps
    def bad_decorator(fn):
        def wrapper(*a, **kw): return fn(*a, **kw)
        return wrapper   # no @functools.wraps(fn)

    def good_decorator(fn):
        @functools.wraps(fn)
        def wrapper(*a, **kw): return fn(*a, **kw)
        return wrapper

    @bad_decorator
    def greet_bad(name: str) -> str:
        """Say hello."""
        return f"Hello, {name}"

    @good_decorator
    def greet_good(name: str) -> str:
        """Say hello."""
        return f"Hello, {name}"

    print(f"bad  __name__={greet_bad.__name__!r}  __doc__={greet_bad.__doc__!r}")
    print(f"good __name__={greet_good.__name__!r}  __doc__={greet_good.__doc__!r}")

    # GOTCHA 3: stacking order
    def mark(label):
        def deco(fn):
            @functools.wraps(fn)
            def w(*a,**kw): return f"[{label}:{fn(*a,**kw)}]"
            return w
        return deco

    @mark("outer")
    @mark("inner")
    def base(): return "core"

    # Applied bottom-up: inner first, then outer wraps inner
    # Executed top-down: outer runs first, then calls inner
    print(f"Stacked result: {base()!r}")   # '[outer:[inner:core]]'

    # GOTCHA 4: parameterised decorator must be CALLED even with defaults
    def retry(times=3):
        def deco(fn):
            @functools.wraps(fn)
            def w(*a,**kw):
                for _ in range(times):
                    try: return fn(*a,**kw)
                    except Exception: pass
                raise RuntimeError("all retries failed")
            return w
        return deco

    # WRONG (if times defaults, you might try @retry without parentheses)
    # @retry        ← passes the function object as `times`; deco receives fn not an int!
    # RIGHT:
    # @retry()      ← calls retry(), returns deco, which then wraps the function
    # @retry(5)     ← calls retry(5), returns deco, which wraps

    count = [0]
    @retry(times=3)
    def flaky():
        count[0] += 1
        if count[0] < 3: raise ValueError("retry me")
        return "ok"

    result = flaky()
    print(f"retry(times=3) succeeded on attempt {count[0]}: {result!r}")


def run_descriptor_notebook() -> None:
    notebook_descriptor_gotchas()
    notebook_slots_gotchas()
    notebook_metaclass_gotchas()
    notebook_decorator_factory_gotchas()
    print("\n" + "═"*64)
    print("  DESCRIPTOR/SLOTS/METACLASS NOTEBOOK COMPLETE")
    print("═"*64)


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_descriptor_notebook()

---
## Typing, Protocols & Memory Model


# Typing And Memory

*Run each cell with **Shift+Enter***

00 — Python Mastery: Typing, Custom Context Managers & Memory Model
===================================================================

Runnable companion to PDF Book I "Advanced Python — types, resources, memory".

Three senior-level topics:

  * TYPING     — type hints don't change runtime behavior but power tooling
                 (mypy/Pylance), self-document, and enable generics/Protocols
                 (structural "duck" typing you can check).
  * CONTEXT    — the @contextmanager decorator is the concise way to write a
    MANAGERS     with-block resource guard (setup / yield / teardown), and
                 contextlib.suppress ignores chosen exceptions.
  * MEMORY     — CPython frees objects by REFERENCE COUNTING (immediate) plus a
                 cyclic GC for reference cycles. weakref lets you reference an
                 object WITHOUT keeping it alive (great for caches).

In [ ]:
import gc
import sys
import weakref
from contextlib import contextmanager, suppress
from dataclasses import dataclass
from typing import Generic, Protocol, TypeVar

---
## TYPING: a generic stack + a Protocol (structural typing). These are checked


In [ ]:
T = TypeVar("T")


class Stack(Generic[T]):
    """A type-safe stack: Stack[int] vs Stack[str] are distinct to a type checker."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def __len__(self) -> int:
        return len(self._items)


class SupportsArea(Protocol):
    """Structural typing: ANY object with .area() -> float satisfies this,
    no inheritance required (duck typing you can statically verify)."""

    def area(self) -> float: ...


@dataclass
class Circle:
    r: float

    def area(self) -> float:
        return 3.14159 * self.r * self.r


@dataclass
class Square:
    side: float

    def area(self) -> float:
        return self.side * self.side


def total_area(shapes: list[SupportsArea]) -> float:
    return sum(s.area() for s in shapes)


def typing_demo() -> None:
    s: Stack[int] = Stack()
    s.push(1)
    s.push(2)
    assert len(s) == 2
    assert s.pop() == 2
    # Circle and Square share NO base class, yet both satisfy SupportsArea.
    shapes = [Circle(2), Square(3)]
    assert abs(total_area(shapes) - (3.14159 * 4 + 9)) < 1e-6
    print("   typing: generic Stack[T] + Protocol (structural) — checked statically, correct at runtime")

---
## CUSTOM CONTEXT MANAGER via @contextmanager: setup -> yield -> teardown, with


In [ ]:
@contextmanager
def timing(log: list):
    log.append("enter")
    try:
        yield log                       # value bound to `as`
    finally:
        log.append("exit")              # always runs (like __exit__)


def context_manager_demo() -> None:
    log: list = []
    with timing(log) as l:
        l.append("body")
    assert log == ["enter", "body", "exit"]

    log2: list = []
    with suppress(ZeroDivisionError):   # swallow a specific exception, tidily
        with timing(log2):
            _ = 1 / 0                    # raises, but teardown still runs
    assert log2 == ["enter", "exit"]     # 'exit' proves finally ran on error
    print("   context manager: @contextmanager (setup/yield/teardown) + suppress() for chosen errors")

---
## MEMORY: reference counting frees objects immediately when the last reference


In [ ]:
def refcount_demo() -> None:
    a = ["x", "y", "z"]
    base = sys.getrefcount(a)            # +1 temporary ref from the call itself
    b = a                               # new reference to the same list
    assert sys.getrefcount(a) == base + 1
    del b                               # drop it again
    assert sys.getrefcount(a) == base
    print("   refcount: each new binding raises the count; dropping it lowers it (immediate free at 0)")


def cyclic_gc_demo() -> None:
    class Node:
        def __init__(self):
            self.ref = None

    # Build a cycle: refcounting alone can NEVER free this (each keeps the other alive).
    x, y = Node(), Node()
    x.ref = y
    y.ref = x
    wx = weakref.ref(x)                  # observe without keeping alive
    del x, y                            # refcounts still > 0 due to the cycle
    collected = gc.collect()            # the cyclic collector breaks it
    assert wx() is None, "cyclic GC should have reclaimed the cycle"
    assert collected >= 0
    print("   cyclic GC: reference cycles are reclaimed by gc.collect(), not by refcounting")


def weakref_demo() -> None:
    class Big:
        pass

    obj = Big()
    cache = weakref.WeakValueDictionary()
    cache["k"] = obj                    # cache does NOT keep obj alive on its own
    assert cache.get("k") is obj
    del obj                             # last strong ref gone
    assert cache.get("k") is None       # entry vanished automatically — no leak
    print("   weakref: a WeakValueDictionary cache never keeps its values alive (leak-free cache)")


def main() -> None:
    print("=" * 70)
    print("PYTHON MASTERY — typing_and_memory.py")
    print("=" * 70)
    print("1. Typing (generics + Protocol/structural typing):")
    typing_demo()
    print("2. Custom context managers (@contextmanager + suppress):")
    context_manager_demo()
    print("3. Reference counting:")
    refcount_demo()
    print("4. Cyclic garbage collection:")
    cyclic_gc_demo()
    print("5. weakref (leak-free caches):")
    weakref_demo()
    print("-" * 70)
    print("All typing_and_memory demos passed ✔")

## ═══  EXHAUSTIVE GOTCHA NOTEBOOK  ══════════════════════════════════════════

In [ ]:
import sys as _sys, gc, tracemalloc
from typing import Any, Callable, ClassVar, Literal, TypeVar, Union, overload

def _sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")


def notebook_typing_gotchas() -> None:

## §T1 · Type Hints Are NOT Runtime Validation

In [ ]:
# Type hints are transparent at runtime.
    # GOTCHA 1: Python happily accepts wrong types at runtime; hints are for TOOLS.
    # GOTCHA 2: Optional[X] == Union[X, None]; X | None is the modern syntax.
    # GOTCHA 3: TypeVar bound vs constrained — very different semantics.
    # GOTCHA 4: Covariance/contravariance — a common source of mypy errors.
    # GOTCHA 5: TYPE_CHECKING guard for imports only needed by type checker.

    # GOTCHA 1: no runtime enforcement
    def add(a: int, b: int) -> int:
        return a + b   # type: ignore

    result = add("hello", " world")   # string, not int — Python doesn't care at runtime
    print(f"GOTCHA 1 — type hints ignored at runtime: {result!r}")   # 'hello world'

    # GOTCHA 2: Optional[X] vs X | None
    from typing import Optional
    def greet(name: Optional[str] = None) -> str:   # same as str | None
        return f"Hello, {name or 'stranger'}"
    print(greet())    # 'Hello, stranger'
    print(greet("Ada"))  # 'Hello, Ada'

    # GOTCHA 3: TypeVar BOUND vs CONSTRAINED
    from typing import TypeVar
    T_bound = TypeVar("T_bound", bound=int)  # T_bound must be int or subclass of int
    T_const = TypeVar("T_const", int, str)   # T_const must be exactly int OR exactly str

    # With bound: the type is preserved (e.g. bool stays bool)
    def double_bound(x: T_bound) -> T_bound:
        return x + x  # type: ignore

    # With constrained: mypy may widen to Union[int, str]
    def double_const(x: T_const) -> T_const:
        return x + x  # type: ignore

    print(f"bound TypeVar: double_bound(True) = {double_bound(True)!r}  type={type(double_bound(True)).__name__}")

    # GOTCHA 4: Literal type — exact values only
    from typing import Literal
    Direction = Literal["north","south","east","west"]
    def move(d: Direction) -> str:
        return f"moving {d}"
    # mypy would flag: move("up")  ← not a valid Literal

    # GOTCHA 5: TYPE_CHECKING
    from typing import TYPE_CHECKING
    if TYPE_CHECKING:
        from collections.abc import Sequence  # never actually imported at runtime
    print("TYPE_CHECKING block only runs during static analysis, not at runtime")

    # Callable type hint
    from typing import Callable
    def apply(fn: Callable[[int, int], int], a: int, b: int) -> int:
        return fn(a, b)
    print(f"apply(lambda a,b:a+b, 3, 4) = {apply(lambda a,b:a+b, 3, 4)}")


def notebook_memory_gotchas() -> None:

## §T2 · CPython Memory Model Gotchas

In [ ]:
# GOTCHA 1: __del__ is UNRELIABLE — called by GC, not deterministically.
    #   Avoid __del__ for resource cleanup; use context managers instead.
    # GOTCHA 2: gc.collect() is O(heap); calling it in a tight loop is expensive.
    # GOTCHA 3: Large local variables captured in a closure are NOT freed
    #   while the closure is alive.
    # GOTCHA 4: sys.getrefcount() always returns at least 1 (the function call itself).

    # GOTCHA 1: __del__ unreliability
    destroyed = []
    class Fragile:
        def __del__(self):
            destroyed.append("destroyed")

    f = Fragile()
    del f             # refcount → 0 → __del__ called immediately in CPython
    gc.collect()
    print(f"GOTCHA 1: __del__ called? {len(destroyed)>0}  (CPython: yes; other impls: maybe not)")

    # GOTCHA 4: sys.getrefcount adds 1
    x = [1, 2, 3]
    count = _sys.getrefcount(x)   # +1 for the arg to getrefcount
    print(f"getrefcount([1,2,3]) = {count}  (at least 2: x + call arg)")

    # GOTCHA 3: closure captures large object
    def make_closure():
        large_data = list(range(100_000))   # 100K ints
        def inner():
            return large_data[0]   # closure keeps large_data alive!
        return inner

    closure = make_closure()
    # large_data is NOT freed even though make_closure() returned —
    # the closure `inner` holds a reference to it.
    print(f"Closure keeps large_data alive: inner()={closure()}")
    # Fix: del large_data inside make_closure after building the closure,
    # or only capture what you need.

    # tracemalloc: find what's using memory
    tracemalloc.start()
    waste = [bytearray(1024) for _ in range(100)]  # allocate 100 KB
    snap = tracemalloc.take_snapshot()
    top = snap.statistics("lineno")[:2]
    for stat in top:
        print(f"  tracemalloc: {stat}")
    del waste
    tracemalloc.stop()


def notebook_weakref_gotchas() -> None:

## §T3 · weakref Gotchas

In [ ]:
# GOTCHA 1: weakref cannot be used on MOST built-in types (int, str, list, dict, tuple).
    # GOTCHA 2: After the referent is GC'd, calling the weakref returns None.
    #   Forgetting to check leads to AttributeError on None.
    # GOTCHA 3: WeakValueDictionary: entries vanish asynchronously.
    #   Between len() and access, entries may disappear.
    # GOTCHA 4: WeakSet vs WeakValueDictionary vs WeakKeyDictionary — choose carefully.

    # GOTCHA 1: built-in types don't support weakref
    for builtin in [42, "hello", [1,2], {"a":1}]:
        try:
            weakref.ref(builtin)
        except TypeError as e:
            print(f"GOTCHA 1 — weakref({type(builtin).__name__}): {e}")

    # Custom classes support weakref by default (unless __slots__ without __weakref__)
    class MyObj:
        def __init__(self, v): self.v = v

    obj = MyObj(99)
    ref = weakref.ref(obj)
    print(f"Live ref:   {ref()!r}")   # MyObj(v=99)
    del obj
    print(f"After del:  {ref()!r}")   # None — referent is gone

    # GOTCHA 2: forgetting to check None
    # ref()  may return None; calling .v on None → AttributeError
    val = ref()
    if val is None:
        print("GOTCHA 2: must check ref() != None before using it")
    else:
        print(val.v)

    # GOTCHA 3: WeakValueDictionary is NOT thread-safe against size changes
    cache = weakref.WeakValueDictionary()
    objects = [MyObj(i) for i in range(5)]
    for i, o in enumerate(objects):
        cache[i] = o

    print(f"cache size before del: {len(cache)}")
    del objects[2]   # MyObj(2) may be GC'd
    gc.collect()
    # After GC, key 2 may be gone — don't iterate and modify simultaneously
    remaining = {k: v.v for k,v in cache.items() if v is not None}
    print(f"cache after del objects[2]: keys={list(remaining.keys())}")

    # WeakKeyDictionary: keys are weak (entry disappears when key is GC'd)
    weak_key_cache = weakref.WeakKeyDictionary()
    k1 = MyObj("key1")
    weak_key_cache[k1] = "metadata"
    print(f"WeakKeyDictionary: {weak_key_cache[k1]!r}")
    del k1
    gc.collect()
    print(f"After del key: len={len(weak_key_cache)}")   # 0


def notebook_gc_internals() -> None:

## §T4 · Cyclic GC — Internals & Tuning

In [ ]:
# CPython uses three GENERATIONS (0, 1, 2).
    # Generation 0: youngest objects; collected most often.
    # Generation 2: oldest survivors; collected rarely.
    # gc.get_count() → (gen0, gen1, gen2) counts.
    # gc.get_threshold() → (700, 10, 10) defaults.
    #
    # GOTCHA 1: Long-lived caches fill generation 2; periodic gc.collect(2) helps.
    # GOTCHA 2: gc.disable() speeds up allocation-heavy code but cycles never free.
    # GOTCHA 3: Objects with __del__ methods move to gc.garbage if they're in a cycle
    #           (Python < 3.4). In 3.4+ this is fixed.

    print(f"GC generations (count): {gc.get_count()}")
    print(f"GC thresholds:          {gc.get_threshold()}")
    print(f"GC enabled:             {gc.isenabled()}")

    # Create and collect a cycle
    class Node:
        def __init__(self): self.ref = None

    x, y = Node(), Node()
    x.ref, y.ref = y, x
    rx = weakref.ref(x)
    del x, y
    before = gc.collect()   # force collection
    print(f"Objects collected:  {before}")
    print(f"Cycle freed:        {rx() is None}")   # True


def run_typing_notebook() -> None:
    notebook_typing_gotchas()
    notebook_memory_gotchas()
    notebook_weakref_gotchas()
    notebook_gc_internals()
    print("\n" + "═"*64)
    print("  TYPING & MEMORY NOTEBOOK COMPLETE")
    print("═"*64)


if __name__ == "__main__":
    if hasattr(_sys.stdout, "reconfigure"):
        _sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_typing_notebook()